# fedOPD · Improvement programme — DR1–DR4 (Tier 1–4), prepared, not yet run

Pre-registered in `CLAUDE.md` §11 / `docs/WORKLOG.md` (2026-07-16). This notebook implements the whole
ladder; **running it later** executes everything top-to-bottom on CPU and writes one JSON per run to
`results/improve/`.

| DR | Tier | What |
|---|---|---|
| DR1 | Tier 1 · optimization | (1) 100–200 FL rounds + server optimizer (FedAdam/FedAvgM) · (2) naive-**blend** ŷ=α·model+(1−α)·naive, α on train-tail · (3) loss↔metric alignment (MAE/Huber vs MSE) |
| DR2 | Tier 2 · new features (same dataset) | (4) flow-state features (lagged backlog/completed/>18wk share) · (5) national+ICB per-dept demand at t−1 (public → FL-safe) · (6) working-days / bank-holidays per month |
| DR3 | Tier 3 · model & evaluation | (7) ridge-AR + gradient-boosted comparators · (8) direct multi-horizon 1–12 mo · (9) quantile forecasts (P50/P90) + agentic planning vs P90 |
| DR4 | Tier 4 · external data | region lookup (ODS API), GP-referrals / population-IMD scaffolds (guarded, off by default) |

**Protocol (unchanged from the build):** every config is scored on the SAME frozen protocol — the
18,191 (series,month) test points, sMAPE on raw counts, macro-mean across the 1,528 eval series —
against the frozen naive bar **16.34** (median 12.92). Reference to beat: fedavg+ft **16.24 ± 0.07**.
Nothing is claimed inside seed noise. Gates are blocking: the extended builder must reproduce the
frozen tensors exactly when all new features are switched OFF.

In [ ]:
# Configuration — flip QUICK=False for the full 5-seed programme.
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import sys, json, time, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "CLAUDE.md").exists())
sys.path.insert(0, str(ROOT))
from src.fedopd import data as fdata, fed as ffed, evaluate as feval
from src.fedopd.model import Forecaster, clone_with

torch.set_num_threads(2)
OUT = ROOT / "results" / "improve"; OUT.mkdir(parents=True, exist_ok=True)

QUICK        = True                      # True: seed 0 only (smoke) · False: full programme
SEEDS        = [0] if QUICK else [0, 1, 2, 3, 4]
ROUNDS_BASE  = 40                        # the build-v1 reference setting
ROUNDS_LONG  = 150                       # DR1(1): longer training
RUN_DR12     = True                      # the config ladder (DR1+DR2)
RUN_DR3      = True                      # comparators, multi-horizon, quantiles
RUN_DR4      = False                     # external data — needs internet; off by default
BAR          = 16.34                     # frozen naive-t1 macro-mean (results/baselines.json)

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                              text=True, cwd=ROOT).stdout.strip()
    except Exception:
        return "?"

print(f"QUICK={QUICK} seeds={SEEDS} | out -> {OUT}")

## DR2(6) · Working days per month (England bank holidays)
Monthly counts are mechanically driven by the number of working days (Easter moves; Dec/Apr/Aug
troughs). Deterministic calendar arithmetic — no leakage possible. Minor errors in the holiday rules
only perturb a count feature; the one-off holidays 2020–2023 are included explicitly.

In [ ]:
from dateutil.easter import easter
import datetime as dt

def _obs(d, taken):
    """Substitute day: weekend (or collision) rolls to the next free weekday."""
    while d.weekday() >= 5 or d in taken:
        d += dt.timedelta(days=1)
    return d

def england_bank_holidays(year):
    hols = set()
    def add(d): hols.add(_obs(d, hols))
    add(dt.date(year, 1, 1))                                     # New Year
    e = easter(year)
    hols.add(e - dt.timedelta(days=2)); hols.add(e + dt.timedelta(days=1))   # Good Fri, Easter Mon
    may = dt.date(year, 5, 1)
    early_may = may + dt.timedelta(days=(7 - may.weekday()) % 7)             # first Monday of May
    if year == 2020: early_may = dt.date(2020, 5, 8)                         # VE-day swap
    hols.add(early_may)
    spring = dt.date(year, 5, 31)
    while spring.weekday() != 0: spring -= dt.timedelta(days=1)              # last Monday of May
    if year == 2022: spring = dt.date(2022, 6, 2)                            # moved for Jubilee
    hols.add(spring)
    summer = dt.date(year, 8, 31)
    while summer.weekday() != 0: summer -= dt.timedelta(days=1)              # last Monday of Aug
    hols.add(summer)
    add(dt.date(year, 12, 25)); add(dt.date(year, 12, 26))                   # Christmas, Boxing
    for one_off in [(2022, 6, 3), (2022, 9, 19), (2023, 5, 8)]:              # Jubilee, funeral, coronation
        if one_off[0] == year: hols.add(dt.date(*one_off))
    return hols

_HOLS = {y: england_bank_holidays(y) for y in range(2017, 2027)}

def workdays_in_month(ts):
    """Weekdays minus England bank holidays for the month of Timestamp ts."""
    d0 = ts.date().replace(day=1)
    nxt = (d0.replace(year=d0.year + 1, month=1) if d0.month == 12
           else d0.replace(month=d0.month + 1))
    n, d = 0, d0
    while d < nxt:
        if d.weekday() < 5 and d not in _HOLS[d.year]:
            n += 1
        d += dt.timedelta(days=1)
    return n

# sanity (no data touched): typical month has 19-23 working days
_w = [workdays_in_month(pd.Timestamp(f"{y}-{m:02d}-01")) for y in (2019, 2023) for m in range(1, 13)]
assert 17 <= min(_w) and max(_w) <= 23, _w
print("workdays sanity:", min(_w), "..", max(_w), "days/month")

## DR2 · Extended dataset builder
Reuses the frozen panel logic (`fdata.freeze_panel`) and appends new feature blocks **after** the
12 lags + 4 calendar/regime slots, so the delta anchor (lag-1 at column 11) is untouched:

`[12 scaled lags | sin cos covid strike | wd(t) wd(t−1) | B(t−1) C(t−1) breach%(t−1) flow_miss | nat(t−1) icb(t−1) | 17 dept one-hot]`

**Leakage rules (identical discipline to the build):** every new feature at target month *t* uses only
values from *t−1* or earlier; all scalers fit on train months (≤ 2025-03) only; flow/national scalers
per series / per dept respectively. **GATE:** with every switch OFF the builder must reproduce the
frozen tensors bit-for-bit (108065×33 / 18191×33, R-01 = 16.34).

In [ ]:
TRAIN_END = pd.Timestamp("2025-03-01")
W = 12

def build_extended(use_workdays=False, use_flow=False, use_national=False,
                   depts_mode="stable18", min_coverage=0.5, panel_min_months=96):
    """data.build_tensors re-implemented with optional feature blocks. Returns the same dict as
    fdata.build_all plus tr_month (target month per train row) for tail fitting."""
    canon = fdata.load_canonical()
    panel = fdata.freeze_panel(canon, depts_mode, min_coverage, panel_min_months, w=W)
    months, roster = panel["months"], panel["roster"]
    reg = pd.read_csv(ROOT / "reports" / "tables" / "07_outlier_register.csv", parse_dates=["date"])
    flagged = set(zip(reg.branch, reg.dept, reg.date))
    dem = canon.pivot_table("demand", ["branch", "dept"], "date").reindex(columns=months)

    mo = np.array([m.month for m in months])
    msin, mcos = np.sin(2 * np.pi * mo / 12), np.cos(2 * np.pi * mo / 12)
    covid = ((months >= "2020-03-01") & (months <= "2021-03-01")).astype(float)
    strike = ((months >= "2022-12-01") & (months <= "2023-12-01")).astype(float)
    ridx = {d: i for i, d in enumerate(roster)}; nr = len(roster)
    tr_t = [i for i in range(W, len(months)) if months[i] <= TRAIN_END]
    te_t = [i for i in range(W, len(months)) if months[i] > TRAIN_END]
    tr_mask_m = months <= TRAIN_END

    # --- optional blocks, all lag-1 and train-scaled ------------------------------------
    wd = np.array([workdays_in_month(m) for m in months], float)
    wd_z = (wd - wd[tr_mask_m].mean()) / max(wd[tr_mask_m].std(), 1e-6)

    if use_flow:
        flow = pd.read_parquet(ROOT / "reports" / "tables" / "rtt_flow.parquet")
        fB = flow.pivot_table("backlog", ["branch", "dept"], "date").reindex(columns=months)
        fC = flow.pivot_table("completed", ["branch", "dept"], "date").reindex(columns=months)
        fS = flow.pivot_table("breach18", ["branch", "dept"], "date").reindex(columns=months)

    if use_national:
        nat = canon.pivot_table("demand", "dept", "date", aggfunc="sum").reindex(columns=months)
        nat_mu = nat.loc[:, tr_mask_m].mean(axis=1); nat_sd = nat.loc[:, tr_mask_m].std(axis=1).clip(lower=1e-6)
        icb_tab = canon.pivot_table("demand", ["icb", "dept"], "date", aggfunc="sum").reindex(columns=months)
        icb_mu = icb_tab.loc[:, tr_mask_m].mean(axis=1); icb_sd = icb_tab.loc[:, tr_mask_m].std(axis=1).clip(lower=1e-6)
        b2icb = canon[["branch", "icb"]].drop_duplicates("branch").set_index("branch")["icb"]

    def fit_scaler(b, d, row):
        keep = [row[i] for i in range(len(months))
                if months[i] <= TRAIN_END and not np.isnan(row[i]) and (b, d, months[i]) not in flagged]
        if not keep: return 0.0, 1.0
        a = np.asarray(keep); return float(a.mean()), max(float(a.std()), 1e-6)

    def flow_feats(b, d, t):
        """log1p-z of backlog/completed at t-1 (train-only stats), breach share, missing flag."""
        out = np.zeros(4)
        try:
            B, Cc, S = fB.loc[(b, d)].values, fC.loc[(b, d)].values, fS.loc[(b, d)].values
        except KeyError:
            out[3] = 1.0; return out
        bl, cl = np.log1p(B), np.log1p(Cc)
        bl_tr, cl_tr = bl[tr_mask_m], cl[tr_mask_m]
        bmu, bsd = np.nanmean(bl_tr), max(np.nanstd(bl_tr), 1e-6)
        cmu, csd = np.nanmean(cl_tr), max(np.nanstd(cl_tr), 1e-6)
        if np.isnan(B[t-1]) or np.isnan(Cc[t-1]):
            out[3] = 1.0; return out
        out[0] = (bl[t-1] - bmu) / bsd
        out[1] = (cl[t-1] - cmu) / csd
        out[2] = (S[t-1] / B[t-1]) if B[t-1] > 0 else 0.0
        return out

    def nat_feats(b, d, t):
        z1 = (nat.loc[d].values[t-1] - nat_mu[d]) / nat_sd[d] if d in nat.index else 0.0
        key = (b2icb.get(b), d)
        z2 = ((icb_tab.loc[key].values[t-1] - icb_mu[key]) / icb_sd[key]
              if key in icb_tab.index else 0.0)
        z1 = 0.0 if np.isnan(z1) else z1; z2 = 0.0 if np.isnan(z2) else z2
        return np.array([z1, z2])

    def make(b, d, row, t, mean, sd, oh):
        parts = [(row[t-W:t] - mean) / sd, [msin[t], mcos[t], covid[t], strike[t]]]
        if use_workdays: parts.append([wd_z[t], wd_z[t-1]])
        if use_flow:     parts.append(flow_feats(b, d, t))
        if use_national: parts.append(nat_feats(b, d, t))
        parts.append(oh)
        return np.concatenate([np.asarray(p, float) for p in parts])

    Xtr, ytr, owner, tr_month = [], [], [], []
    Xte, yte, meta = [], [], []
    scalers = {}
    for b, d in panel["eval_series"] + panel["train_only"]:
        row = dem.loc[(b, d)].values
        mean, sd = fit_scaler(b, d, row); scalers[(b, d)] = (mean, sd)
        oh = np.zeros(nr); oh[ridx[d]] = 1.0
        for t in tr_t:
            if np.isnan(row[t]) or np.isnan(row[t-W:t]).any(): continue
            Xtr.append(make(b, d, row, t, mean, sd, oh)); ytr.append((row[t]-mean)/sd)
            owner.append(b); tr_month.append(months[t])
    for b, d in panel["eval_series"]:
        row = dem.loc[(b, d)].values; mean, sd = scalers[(b, d)]
        oh = np.zeros(nr); oh[ridx[d]] = 1.0
        for t in te_t:
            if np.isnan(row[t]) or np.isnan(row[t-W:t]).any(): continue
            Xte.append(make(b, d, row, t, mean, sd, oh)); yte.append((row[t]-mean)/sd)
            meta.append((b, d, str(months[t].date()), mean, sd, float(row[t]), float(row[t-1])))

    Xtr = np.asarray(Xtr, np.float32); ytr = np.asarray(ytr, np.float32)
    Xte = np.asarray(Xte, np.float32); yte = np.asarray(yte, np.float32)
    te_df = pd.DataFrame(meta, columns=["branch","dept","month","mean","sd","actual","naive_raw"])
    ds = dict(Xtr=Xtr, ytr=ytr, owner=np.asarray(owner), tr_month=pd.DatetimeIndex(tr_month),
              Xte=Xte, yte=yte, te_df=te_df, n_features=Xtr.shape[1], n_lags=W,
              clients=sorted(set(owner)), panel=panel)
    # leakage asserts (same 5 as the build)
    tm = pd.to_datetime(te_df.month)
    assert ((tm >= "2025-04-01") & (tm <= "2026-03-01")).all()
    assert not (np.isnan(Xtr).any() or np.isnan(Xte).any())
    assert set(te_df.branch) <= set(ds["clients"])
    return ds

def parity_gate():
    """GATE: all switches OFF must reproduce the frozen build tensors exactly."""
    ref, ext = fdata.build_all(), build_extended()
    assert ext["Xtr"].shape == ref["Xtr"].shape == (108065, 33), (ext["Xtr"].shape, ref["Xtr"].shape)
    assert ext["Xte"].shape == ref["Xte"].shape
    assert np.allclose(ext["Xtr"], ref["Xtr"], atol=1e-5) and np.allclose(ext["Xte"], ref["Xte"], atol=1e-5)
    bar = feval.naive_bar(ext["te_df"]); assert abs(bar["macro_mean"] - BAR) < 0.01, bar
    print("PARITY GATE PASS — extended builder reproduces the frozen tensors; bar", round(bar["macro_mean"], 2))

parity_gate()

## DR1 · Optimization machinery
(1) **Server optimizers.** Plain FedAvg replaces the global model with the weighted client mean.
FedAvgM/FedAdam instead treat (mean − global) as a pseudo-gradient with server momentum / Adam —
standard remedies for the slow tail we saw (loss still descending at round 40).
(2) **Naive-blend.** ŷ = α·model + (1−α)·naive with a single global α fitted by closed-form least
squares on the train-tail (targets in the last 12 train months). Hard *selection* failed (16.83);
blending is its noise-robust relaxation and anchors the volatile small series.
(3) **Loss alignment.** We score sMAPE on raw counts but trained MSE on z-scores; MAE/Huber are less
tail-dominated and closer to the metric.

In [ ]:
class LossForecaster(Forecaster):
    """Forecaster with configurable training loss: mse | mae | huber | pinball-q."""
    def __init__(self, *a, loss="mse", q=0.5, **kw):
        super().__init__(*a, **kw); self.loss_name, self.q = loss, q

    def _loss(self, pred, tgt):
        e = tgt - pred
        if self.loss_name == "mae":    return e.abs().mean()
        if self.loss_name == "huber":  return torch.nn.functional.huber_loss(pred, tgt, delta=1.0)
        if self.loss_name == "pinball": return torch.maximum(self.q * e, (self.q - 1) * e).mean()
        return (e ** 2).mean()

    def train_epochs(self, X, y, epochs=1, lr=1e-3, batch=64, mu=0.0, global_params=None,
                     val=None, patience=5):
        X = torch.from_numpy(np.asarray(X, np.float32))
        t = torch.from_numpy(self._target(np.asarray(X), np.asarray(y, np.float32)))
        opt = torch.optim.Adam(self.net.parameters(), lr=lr, weight_decay=self.weight_decay)
        gref = ([torch.from_numpy(np.asarray(w, np.float32)) for w in global_params]
                if (mu > 0 and global_params is not None) else None)
        g = torch.Generator().manual_seed(self.seed); loss = torch.tensor(0.0)
        for _ in range(epochs):
            for idx in torch.randperm(len(X), generator=g).split(batch):
                opt.zero_grad()
                loss = self._loss(self.net(X[idx]).squeeze(-1), t[idx])
                if gref is not None:
                    loss = loss + (mu / 2) * sum(((p - w) ** 2).sum()
                                                 for p, w in zip(self.net.parameters(), gref))
                loss.backward(); opt.step(); loss = loss.detach()
        return float(loss)

def factory(nf, target, seed, loss="mse", q=0.5):
    def make():
        return (Forecaster(nf, target, seed) if loss == "mse"
                else LossForecaster(nf, target, seed, loss=loss, q=q))
    return make

def run_federated_v2(cd, nf, target="delta", rounds=40, lr=1e-3, seed=0,
                     server_opt="avg", server_lr=0.5, beta1=0.9, beta2=0.99, tau=1e-3,
                     loss="mse"):
    """FedAvg with optional FedAvgM / FedAdam server step. Returns (params, history)."""
    mk = factory(nf, target, seed, loss)
    clients = sorted(cd); sizes = np.array([len(cd[c][1]) for c in clients], float)
    wts = sizes / sizes.sum()
    glob = mk().get_params()
    m = [np.zeros_like(w) for w in glob]; v = [np.zeros_like(w) for w in glob]
    hist = []
    for r in range(rounds):
        ups, ls = [], []
        for i, c in enumerate(clients):
            cm = mk(); cm.set_params(glob)
            ls.append(cm.train_epochs(*cd[c], epochs=1, lr=lr))
            ups.append(cm.get_params())
        avg = [np.sum([wts[i] * u[k] for i, u in enumerate(ups)], axis=0) for k in range(len(glob))]
        if server_opt == "avg":
            glob = avg
        else:
            delta = [a - g for a, g in zip(avg, glob)]
            m = [beta1 * mi + (1 - beta1) * di for mi, di in zip(m, delta)]
            if server_opt == "fedavgm":
                glob = [g + server_lr * mi for g, mi in zip(glob, m)]
            else:                                      # fedadam
                v = [beta2 * vi + (1 - beta2) * di ** 2 for vi, di in zip(v, delta)]
                glob = [g + server_lr * mi / (np.sqrt(vi) + tau) for g, mi, vi in zip(glob, m, v)]
        hist.append(float(np.mean(ls)))
    return glob, hist

def personalise_v2(glob, cd, ds, nf, target="delta", ft_epochs=3, lr=1e-3, seed=0, loss="mse"):
    """Per-client fine-tune with early stopping on the true chronological tail (tr_month-based)."""
    tail_cut = TRAIN_END - pd.DateOffset(months=11)
    out = {}
    for c, (X, y) in cd.items():
        mask = (ds["owner"] == c)
        tm = ds["tr_month"][mask]
        tail = (tm >= tail_cut)
        cm = factory(nf, target, seed, loss)(); cm.set_params(glob)
        if tail.sum() >= 6 and (~tail).sum() >= 24:
            cm.train_epochs(X[~tail], y[~tail], epochs=ft_epochs, lr=lr,
                            val=(X[tail], y[tail]), patience=2)
        else:
            cm.train_epochs(X, y, epochs=ft_epochs, lr=lr)
        out[c] = cm
    return out

def fit_blend_alpha(models, ds):
    """Global α for  ŷ = α·model + (1−α)·naive  by least squares on the train-tail (scaled space)."""
    tail_cut = TRAIN_END - pd.DateOffset(months=11)
    num = den = 0.0
    for c, mdl in models.items():
        mask = (ds["owner"] == c) & (ds["tr_month"] >= tail_cut).values
        if mask.sum() == 0: continue
        X, y = ds["Xtr"][mask], ds["ytr"][mask]
        p = mdl.predict(X) if mdl is not None else X[:, 11]
        n = X[:, 11]                                   # scaled naive = lag-1 column
        num += float(np.dot(p - n, y - n)); den += float(np.dot(p - n, p - n))
    return float(np.clip(num / max(den, 1e-12), 0.0, 1.0))

def predict_models(models, ds):
    return ffed.predict_per_client(models, ds["te_df"], ds["Xte"], lag1=11)

print("DR1 machinery defined (LossForecaster, run_federated_v2, personalise_v2, blend)")

## The experiment ladder (DR1 + DR2 combined)
One runner; a ladder of configs that adds one lever at a time so each contribution is attributable.
Every run writes `results/improve/<name>_s<seed>.json` with config, metrics (model / +ft / +blend),
its own recomputed bar, seed, git sha, duration. Existing JSONs are skipped (resumable).

In [ ]:
def run_config(name, seed, builder_kw=None, rounds=ROUNDS_BASE, server_opt="avg",
               server_lr=0.5, loss="mse", ft_epochs=3, blend=True, _cache={}):
    out_f = OUT / f"{name}_s{seed}.json"
    if out_f.exists():
        print("skip", out_f.name); return json.loads(out_f.read_text())
    t0 = time.time()
    key = json.dumps(builder_kw or {}, sort_keys=True)
    if key not in _cache:                                   # cache dataset builds across seeds
        _cache[key] = build_extended(**(builder_kw or {}))
    ds = _cache[key]; cd = {}
    for c in ds["clients"]:
        mask = ds["owner"] == c
        cd[c] = (ds["Xtr"][mask], ds["ytr"][mask])
    nf = ds["n_features"]
    glob, hist = run_federated_v2(cd, nf, rounds=rounds, seed=seed,
                                  server_opt=server_opt, server_lr=server_lr, loss=loss)
    gm = factory(nf, "delta", seed, loss)(); gm.set_params(glob)
    rec = {"name": name, "seed": seed, "git": git_sha(),
           "config": dict(builder_kw=builder_kw or {}, rounds=rounds, server_opt=server_opt,
                          server_lr=server_lr, loss=loss, ft_epochs=ft_epochs, blend=blend,
                          n_features=nf),
           "bar_R01": feval.naive_bar(ds["te_df"]),
           "metrics_global": feval.score_scaled(ds["te_df"], gm.predict(ds["Xte"])),
           "loss_first_last": [round(hist[0], 4), round(hist[-1], 4)]}
    models = personalise_v2(glob, cd, ds, nf, ft_epochs=ft_epochs, seed=seed, loss=loss)
    rec["metrics_ft"] = feval.score_scaled(ds["te_df"], predict_models(models, ds))
    if blend:
        a = fit_blend_alpha(models, ds)
        pb = a * predict_models(models, ds) + (1 - a) * ds["Xte"][:, 11]
        rec["alpha"] = round(a, 3)
        rec["metrics_blend"] = feval.score_scaled(ds["te_df"], pb)
    rec["duration_s"] = round(time.time() - t0, 1)
    out_f.write_text(json.dumps(rec, indent=2))
    best = min(rec["metrics_ft"]["macro_mean"],
               rec.get("metrics_blend", rec["metrics_ft"])["macro_mean"])
    print(f"{name} s{seed}: ft {rec['metrics_ft']['macro_mean']:.2f}"
          + (f" | blend(a={rec['alpha']}) {rec['metrics_blend']['macro_mean']:.2f}" if blend else "")
          + f" | bar {rec['bar_R01']['macro_mean']:.2f} | {rec['duration_s']}s")
    return rec

LADDER = [
    # DR1 — optimization on the frozen 33 features
    ("L0_ref40",        dict()),                                                        # reproduces build-v1
    ("L1_rounds150",    dict(rounds=ROUNDS_LONG)),
    ("L1_fedadam",      dict(rounds=ROUNDS_LONG, server_opt="fedadam", server_lr=0.05)),
    ("L1_fedavgm",      dict(rounds=ROUNDS_LONG, server_opt="fedavgm", server_lr=0.7)),
    ("L1_mae",          dict(rounds=ROUNDS_LONG, server_opt="fedadam", server_lr=0.05, loss="mae")),
    ("L1_huber",        dict(rounds=ROUNDS_LONG, server_opt="fedadam", server_lr=0.05, loss="huber")),
    # DR2 — features, added on the best DR1 optimizer (fedadam), one block at a time then all
    ("L2_workdays",     dict(rounds=ROUNDS_LONG, server_opt="fedadam", server_lr=0.05,
                             builder_kw=dict(use_workdays=True))),
    ("L2_flow",         dict(rounds=ROUNDS_LONG, server_opt="fedadam", server_lr=0.05,
                             builder_kw=dict(use_flow=True))),
    ("L2_national",     dict(rounds=ROUNDS_LONG, server_opt="fedadam", server_lr=0.05,
                             builder_kw=dict(use_national=True))),
    ("L2_all",          dict(rounds=ROUNDS_LONG, server_opt="fedadam", server_lr=0.05,
                             builder_kw=dict(use_workdays=True, use_flow=True, use_national=True))),
    ("L2_all_huber",    dict(rounds=ROUNDS_LONG, server_opt="fedadam", server_lr=0.05, loss="huber",
                             builder_kw=dict(use_workdays=True, use_flow=True, use_national=True))),
]

if RUN_DR12:
    for name, kw in LADDER:
        for s in SEEDS:
            run_config(name, s, **kw)

## DR3(7) · Comparator model families
Ridge-AR (linear) and gradient-boosted trees on the identical feature matrix and protocol.
Centralised only — the question is whether the MLP is the bottleneck, not a new FL method. If ridge
matches the MLP, architecture is not where the headroom is.

In [ ]:
def run_comparators(builder_kw=None, tag="base"):
    from sklearn.linear_model import Ridge
    from sklearn.ensemble import HistGradientBoostingRegressor
    ds = build_extended(**(builder_kw or {}))
    lag1 = ds["Xtr"][:, 11]
    y_delta_tr = ds["ytr"] - lag1                      # same naive-anchored target
    out = {}
    for nm, mdl in [("ridge", Ridge(alpha=1.0)),
                    ("gbt", HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                                          max_depth=None, random_state=0))]:
        mdl.fit(ds["Xtr"], y_delta_tr)
        pred = ds["Xte"][:, 11] + mdl.predict(ds["Xte"])
        out[nm] = feval.score_scaled(ds["te_df"], pred)
        print(f"{nm:6s} ({tag}): macro {out[nm]['macro_mean']:.2f} | median {out[nm]['median']:.2f}"
              f" | bar {feval.naive_bar(ds['te_df'])['macro_mean']:.2f}")
    (OUT / f"comparators_{tag}.json").write_text(json.dumps(
        {"tag": tag, "builder_kw": builder_kw or {}, "results": out, "git": git_sha()}, indent=2))
    return out

if RUN_DR3:
    run_comparators(None, "base33")
    run_comparators(dict(use_workdays=True, use_flow=True, use_national=True), "extended")

## DR3(8) · Direct multi-horizon forecasting (1–12 months)
One model, 12 outputs: scaled demand at t..t+11 from the same input window. Needed by the agentic
plane, whose 12-month burn-down currently reuses a 1-month forecast. Train origins must have all 12
targets ≤ 2025-03 (no leakage); the test origin is 2025-04 per series (targets = the whole held-out
year). Reported per horizon; h=1 should be consistent with the single-step models.

In [ ]:
class MultiHorizonNet(torch.nn.Module):
    def __init__(self, nf, H=12):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(nf, 64), torch.nn.ReLU(),
                                       torch.nn.Linear(64, 32), torch.nn.ReLU(),
                                       torch.nn.Linear(32, H))
    def forward(self, x): return self.net(x)

def run_multihorizon(H=12, epochs=40, seed=0):
    """Direct 12-output forecaster on the frozen 33-feature layout. Train origins need all H targets
    <= 2025-03 (no leakage); the single test origin per series is 2025-04 (targets = the held-out year)."""
    torch.manual_seed(seed); np.random.seed(seed)
    canon = fdata.load_canonical()
    panel = fdata.freeze_panel(canon)
    months = panel["months"]; midx = {m: i for i, m in enumerate(months)}
    dem = canon.pivot_table("demand", ["branch", "dept"], "date").reindex(columns=months)
    first_test = midx[pd.Timestamp("2025-04-01")]
    reg = pd.read_csv(ROOT / "reports" / "tables" / "07_outlier_register.csv", parse_dates=["date"])
    flagged = set(zip(reg.branch, reg.dept, reg.date))

    def scaler(b, d, row):
        keep = [row[i] for i in range(len(months))
                if months[i] <= TRAIN_END and not np.isnan(row[i]) and (b, d, months[i]) not in flagged]
        a = np.asarray(keep) if keep else np.array([0.0])
        return float(a.mean()), max(float(a.std()), 1e-6)

    mo = np.array([m.month for m in months])
    msin, mcos = np.sin(2*np.pi*mo/12), np.cos(2*np.pi*mo/12)
    covid = ((months >= "2020-03-01") & (months <= "2021-03-01")).astype(float)
    strike = ((months >= "2022-12-01") & (months <= "2023-12-01")).astype(float)
    roster = panel["roster"]; ridx = {d: i for i, d in enumerate(roster)}

    Xtr, Ytr, Xte, te_meta = [], [], [], []
    for b, d in panel["eval_series"]:
        row = dem.loc[(b, d)].values
        mean, sd = scaler(b, d, row)
        oh = np.zeros(len(roster)); oh[ridx[d]] = 1.0
        for t in range(12, len(months) - H + 1):
            if np.isnan(row[t-12:t]).any() or np.isnan(row[t:t+H]).any(): continue
            x = np.concatenate([(row[t-12:t]-mean)/sd, [msin[t], mcos[t], covid[t], strike[t]], oh])
            if months[t + H - 1] <= TRAIN_END:
                Xtr.append(x); Ytr.append((row[t:t+H]-mean)/sd)
            elif t == first_test:
                Xte.append(x); te_meta.append((b, d, mean, sd, row[t:t+H].copy()))
    Xtr, Ytr, Xte = np.asarray(Xtr, np.float32), np.asarray(Ytr, np.float32), np.asarray(Xte, np.float32)
    print(f"multi-horizon: train {Xtr.shape} | test origins {Xte.shape[0]} series")

    net = MultiHorizonNet(Xtr.shape[1], H)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    Xt, Yt = torch.from_numpy(Xtr), torch.from_numpy(Ytr)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        for idx in torch.randperm(len(Xt), generator=g).split(256):
            opt.zero_grad()
            loss = torch.nn.functional.mse_loss(net(Xt[idx]), Yt[idx]); loss.backward(); opt.step()
    with torch.no_grad():
        P = net(torch.from_numpy(Xte)).numpy()
    per_h = []
    for h in range(H):
        sm = [200*abs(P[i, h]*sd + mean - act[h]) / (abs(P[i, h]*sd + mean) + abs(act[h]) + 1e-6)
              for i, (b, d, mean, sd, act) in enumerate(te_meta)]
        per_h.append(float(np.mean(sm)))
    rec = {"per_horizon_smape": [round(x, 2) for x in per_h], "n_series": len(te_meta),
           "seed": seed, "git": git_sha()}
    (OUT / "multihorizon.json").write_text(json.dumps(rec, indent=2))
    print("sMAPE by horizon:", [round(x, 1) for x in per_h])
    return rec

if RUN_DR3:
    run_multihorizon()

## DR3(9) · Quantile forecasts and P90-based agentic planning
Two pinball-loss models (q=0.5, q=0.9) on the naive-anchored delta target (a quantile of the level is
lag1 + quantile of the delta, since lag1 is a known constant per window). The agentic snapshot is then
recomputed planning against **P90 demand** instead of the mean — the risk-aware capacity plan. Gate:
P90 must be calibrated (≈90% of actuals below it) and RAG counts shift toward RED (more conservative),
with the capacity total rising accordingly.

In [ ]:
def run_quantile_agentic(seed=0):
    """Pinball-loss P50/P90 forecasters (centralised — quantiles need not be federated to prove the
    decision-quality point; a federated variant is a one-line loss switch in run_federated_v2), then
    the agentic snapshot planned against P90 instead of the mean."""
    ds = build_extended()
    te = ds["te_df"]
    preds = {}
    for q in (0.5, 0.9):
        m = LossForecaster(ds["n_features"], "delta", seed, loss="pinball", q=q)
        m.train_epochs(ds["Xtr"], ds["ytr"], epochs=40, lr=1e-3, batch=256)
        preds[q] = m.predict(ds["Xte"]) * te["sd"].values + te["mean"].values
    cover = float(np.mean(te["actual"].values <= preds[0.9]))
    print(f"P90 empirical coverage: {cover:.1%} (target ~90%)  |  P50 macro "
          f"{feval.score_raw(te, preds[0.5])['macro_mean']:.2f} vs bar {feval.naive_bar(te)['macro_mean']:.2f}")

    flow = pd.read_parquet(ROOT / "reports" / "tables" / "rtt_flow.parquet").sort_values(["branch","dept","date"])
    g = flow.groupby(["branch","dept"], group_keys=False)
    flow["B_prev"] = g["backlog"].shift(1)
    flow["C_base"] = g["completed"].transform(lambda s: s.shift(1).rolling(12, min_periods=3).mean())

    def snapshot(dhat, label):
        fc = te.assign(D_hat=np.maximum(0.0, dhat), date=pd.to_datetime(te["month"]))
        dec = fc.merge(flow[["branch","dept","date","B_prev","C_base"]],
                       on=["branch","dept","date"], how="left")
        dec = dec.dropna(subset=["B_prev","C_base"]); dec = dec[dec.C_base > 0]
        s = dec[dec.date == dec.date.max()]
        B_star = (18/4.345) * s["D_hat"]
        gap = (s["D_hat"] + np.maximum(0.0, s["B_prev"] - B_star)/12 - s["C_base"]).clip(lower=0)
        out = {"label": label, "capacity_per_month": int(gap.sum())}
        print(label, out); return out

    rec = {"p90_coverage": round(cover, 3),
           "plan_mean": snapshot(preds[0.5], "plan@P50"),
           "plan_p90": snapshot(preds[0.9], "plan@P90"),
           "seed": seed, "git": git_sha()}
    (OUT / "quantile_agentic.json").write_text(json.dumps(rec, indent=2))
    return rec

if RUN_DR3:
    run_quantile_agentic()

## DR4 · External data (guarded — needs internet; `RUN_DR4=False` by default)
1. **Region lookup** (ODS API): trust code → NHS region, completing the hierarchy tier missing from
   the RTT files. Cached to `results/improve/ods/`.
2. **GP referrals / population+IMD / flu**: scaffolds with source URLs and a clear integration point
   (join keys), to be wired once downloaded. These change the *information set* — the only route past
   the centralised bound.

In [ ]:
def fetch_region_lookup(clients):
    """Trust -> region via the NHS ODS ORD API (cached). Run with internet access."""
    import urllib.request
    cache = OUT / "ods"; cache.mkdir(exist_ok=True)
    out = {}
    for code in clients:
        f = cache / f"{code}.json"
        if not f.exists():
            url = f"https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations/{code}"
            try:
                with urllib.request.urlopen(url, timeout=20) as r:
                    f.write_bytes(r.read())
            except Exception as e:
                print(code, "FAILED:", e); continue
        d = json.loads(f.read_text())["Organisation"]
        rels = d.get("Rels", {}).get("Rel", [])
        region = next((r["Target"]["OrgId"]["extension"] for r in rels
                       if r.get("Target", {}).get("PrimaryRoleId", {}).get("id") == "RO209"
                       and r.get("Status") == "Active"), None)   # RO209 = NHS England region
        out[code] = {"name": d.get("Name"), "region": region,
                     "postcode": d.get("GeoLoc", {}).get("Location", {}).get("PostCode")}
    (OUT / "trust_region_lookup.json").write_text(json.dumps(out, indent=2))
    print(f"region lookup: {sum(1 for v in out.values() if v['region'])}/{len(out)} resolved")
    return out

# --- scaffolds (wire after download; join keys documented) -------------------------------
GP_REFERRALS_SRC = "https://digital.nhs.uk/data-and-information/publications/statistical/mi-pomi"  # monthly
IMD_SRC = "https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019"           # LSOA->trust catchment
FLU_SRC = "https://www.gov.uk/government/statistics/national-flu-and-covid-19-surveillance-reports"

def integrate_external_feature(df_monthly, key_cols, value_col, name):
    """Template: z-score on train months only, lag by 1, merge into build_extended as a new block.
    df_monthly: [date, <key_cols>, value]. Returns the merge-ready frame."""
    df = df_monthly.copy()
    df["date"] = pd.to_datetime(df["date"])
    tr = df[df.date <= TRAIN_END]
    mu, sd = tr[value_col].mean(), max(tr[value_col].std(), 1e-6)
    df[name] = (df[value_col] - mu) / sd
    df["date"] = df["date"] + pd.DateOffset(months=1)      # lag 1: known before the target month
    return df[["date", *key_cols, name]]

if RUN_DR4:
    ds0 = build_extended()
    fetch_region_lookup(ds0["clients"])
else:
    print("DR4 skipped (RUN_DR4=False) — enable with internet access")

## Summary — every run vs the frozen bar
Collects all `results/improve/*.json` into one table. **Honest-claim rule (unchanged):** a config only
"beats" another if the gap exceeds the seed std; with `QUICK=True` (1 seed) treat everything as
provisional and re-run with `QUICK=False` before claiming anything.

In [ ]:
import glob
rows = []
for f in sorted(glob.glob(str(OUT / "*_s*.json"))):
    r = json.loads(open(f).read())
    if "metrics_ft" not in r: continue
    rows.append({"name": r["name"], "seed": r["seed"],
                 "global": round(r["metrics_global"]["macro_mean"], 2),
                 "+ft": round(r["metrics_ft"]["macro_mean"], 2),
                 "+blend": round(r["metrics_blend"]["macro_mean"], 2) if "metrics_blend" in r else None,
                 "alpha": r.get("alpha"), "bar": round(r["bar_R01"]["macro_mean"], 2),
                 "nf": r["config"]["n_features"]})
if rows:
    df = pd.DataFrame(rows)
    agg = (df.groupby("name").agg(ft_mean=("+ft", "mean"), ft_std=("+ft", "std"),
                                  blend_mean=("+blend", "mean"), alpha=("alpha", "mean"),
                                  bar=("bar", "first"), seeds=("seed", "count"), nf=("nf", "first"))
             .round(2).sort_values("blend_mean"))
    print(agg.to_string())
    print(f"\nreference build-v1: fedavg+ft 16.24 ± 0.07 | frozen bar {BAR}")
    print("claim rule: a gap must exceed the seed std; QUICK runs are provisional.")
else:
    print("no ladder results yet — set RUN_DR12=True and run.")

## When this has run
1. Re-run with `QUICK=False` (5 seeds) for any config that looks promising.
2. Log the outcome per gate in `docs/BUILD_LOG.md` (`R-xx` numbers), update the reports if a headline
   changes, and commit with the ids.
3. If `L2_*` beats the bar outside noise → regenerate `fc_methods.png`, the dashboard export, and the
   three PDFs so every artefact tells the same story.
4. DR4 needs internet: set `RUN_DR4=True`, run the region lookup, then wire GP/IMD via
   `integrate_external_feature` as new builder blocks.